In [1]:
# KB
# show the facts and rules for each case
# forward
# backward
# test without cli
# cli

In [2]:
KB = {
    "Vehicle Fault Diagnosis": {
        "facts": ["engine_not_starting", "battery_low", "headlights_dim", "starter_clicking",
                  "engine_overheating", "coolant_low", "oil_low", "vehicle_old"],
        "rules": [
            (["battery_low", "headlights_dim"], "electrical_system_weak"),
            (["battery_low", "starter_clicking"], "electrical_system_weak"),
            (["electrical_system_weak", "engine_not_starting"], "starting_system_fault"),
            (["starting_system_fault", "vehicle_old"], "battery_replacement_required"),
            (["engine_overheating", "coolant_low"], "cooling_system_fault"),
            (["engine_overheating", "oil_low"], "lubrication_system_fault"),
            (["cooling_system_fault", "lubrication_system_fault"], "severe_engine_risk"),
        ],
    },
    "Agriculture / Crop Advisory": {
        "facts": ["soil_dry", "temperature_high", "rainfall_low", "irrigation_irregular",
                  "leaves_yellow", "plant_growth_slow", "leaf_edges_brown", "crop_stage_flowering"],
        "rules": [
            (["soil_dry", "temperature_high"], "severe_water_stress"),
            (["rainfall_low", "irrigation_irregular"], "irrigation_deficiency"),
            (["severe_water_stress", "irrigation_deficiency"], "urgent_irrigation_required"),
            (["leaves_yellow", "plant_growth_slow"], "nutrient_problem"),
            (["leaf_edges_brown", "temperature_high"], "heat_stress"),
            (["heat_stress", "severe_water_stress"], "crop_heat_risk"),
            (["crop_stage_flowering", "severe_water_stress"], "flowering_yield_risk"),
        ],
    },
    "Network Troubleshooting": {
        "facts": ["internet_not_working", "router_off", "power_fluctuation", "wifi_not_visible",
                  "cable_disconnected", "ip_not_assigned", "dns_failure", "multiple_devices_affected"],
        "rules": [
            (["router_off", "power_fluctuation"], "router_power_issue"),
            (["router_power_issue", "wifi_not_visible"], "wireless_service_unavailable"),
            (["cable_disconnected", "internet_not_working"], "physical_connection_failure"),
            (["ip_not_assigned", "multiple_devices_affected"], "dhcp_failure"),
            (["dns_failure", "internet_not_working"], "dns_problem"),
            (["dhcp_failure", "dns_problem"], "network_configuration_failure"),
        ],
    },
    "Cybersecurity Alert System": {
        "facts": ["multiple_failed_logins", "unknown_device", "unusual_location", "multiple_ip_addresses",
                  "admin_account", "sensitive_file_access", "large_data_transfer", "password_changed"],
        "rules": [
            (["multiple_failed_logins", "unknown_device"], "abnormal_login"),
            (["unusual_location", "multiple_ip_addresses"], "suspicious_access"),
            (["abnormal_login", "suspicious_access"], "high_risk_login"),
            (["high_risk_login", "admin_account"], "privileged_account_threat"),
            (["sensitive_file_access", "large_data_transfer"], "data_exfiltration"),
            (["privileged_account_threat", "data_exfiltration"], "critical_security_incident"),
            (["password_changed", "multiple_failed_logins"], "account_compromise"),
        ],
    },
}

In [3]:
def show(case):
    for fs in KB[case]["facts"]:
        print(fs)

    for cond, conc in KB[case]["rules"]:
        print(f"IF {' AND '.join(cond)} THEN {conc}")

In [4]:
show("Cybersecurity Alert System")

multiple_failed_logins
unknown_device
unusual_location
multiple_ip_addresses
admin_account
sensitive_file_access
large_data_transfer
password_changed
IF multiple_failed_logins AND unknown_device THEN abnormal_login
IF unusual_location AND multiple_ip_addresses THEN suspicious_access
IF abnormal_login AND suspicious_access THEN high_risk_login
IF high_risk_login AND admin_account THEN privileged_account_threat
IF sensitive_file_access AND large_data_transfer THEN data_exfiltration
IF privileged_account_threat AND data_exfiltration THEN critical_security_incident
IF password_changed AND multiple_failed_logins THEN account_compromise


In [5]:
def forward(facts, rules):
    known = set(facts)
    step = 0
    changed = True

    while changed:
        changed = False
        for cond, conc in rules:
            if all(c in known for c in cond) and conc not in known:
                known.add(conc)
                changed = True
                step += 1
                print(f"Step {step}: IF {' AND '.join(cond)} THEN {conc}")

    return known - set(facts)

In [6]:
def backward(goal, facts, rules, depth=0):
    intent = " " * depth

    if goal in facts:
        print(f"{intent}{goal} -> known fact")
        return True

    for cond, conc in rules:
        if conc == goal:
            print(f"{intent} trying rule: IF {' AND '.join(cond)} THEN {goal}")
            if all(backward(c, facts, rules, depth+1) for c in cond):
                print(f"{intent}{goal} -> Proved")
                return True

    print(f"{intent}{goal} -> can not be proved")
    return False

In [7]:
case = "Vehicle Fault Diagnosis"
facts = ["battery_low", "starter_clicking", "engine_not_starting", "vehicle_old"]
rules = KB[case]["rules"]

print("Given facts:", facts)
print("\n--- FORWARD CHAINING ---")
print("New facts inferred:", forward(facts, rules))

print("\n--- BACKWARD CHAINING: battery_replacement_required ---")
print("Result:", backward("battery_replacement_required", facts, rules))

Given facts: ['battery_low', 'starter_clicking', 'engine_not_starting', 'vehicle_old']

--- FORWARD CHAINING ---
Step 1: IF battery_low AND starter_clicking THEN electrical_system_weak
Step 2: IF electrical_system_weak AND engine_not_starting THEN starting_system_fault
Step 3: IF starting_system_fault AND vehicle_old THEN battery_replacement_required
New facts inferred: {'starting_system_fault', 'electrical_system_weak', 'battery_replacement_required'}

--- BACKWARD CHAINING: battery_replacement_required ---
 trying rule: IF starting_system_fault AND vehicle_old THEN battery_replacement_required
  trying rule: IF electrical_system_weak AND engine_not_starting THEN starting_system_fault
   trying rule: IF battery_low AND headlights_dim THEN electrical_system_weak
   battery_low -> known fact
   headlights_dim -> can not be proved
   trying rule: IF battery_low AND starter_clicking THEN electrical_system_weak
   battery_low -> known fact
   starter_clicking -> known fact
  electrical_sys

In [ ]:
# cli

cases = list(KB)
for i, c in enumerate(cases, 1):
    print(f"{i}. {c}")
case = cases[int(input("Choose a case: ")) - 1]
rules = KB[case]["rules"]

print("\nFacts:")
for i, f in enumerate(KB[case]["facts"], 1):
    print(f"{i}. {f}")
picked = input("Enter the fact numbers that are true (e.g. 1 2 4): ").split()
facts = [KB[case]["facts"][int(n) - 1] for n in picked]

print("\nGiven facts:", facts)
print("\n--- FORWARD CHAINING ---")
print("New facts inferred:", forward(facts, rules))

goals = sorted({conc for _, conc in rules})
print("\nPossible goals:")
for i, g in enumerate(goals, 1):
    print(f"{i}. {g}")
goal = goals[int(input("Choose a goal to prove: ")) - 1]

print(f"\n--- BACKWARD CHAINING: {goal} ---")
print("Result:", backward(goal, facts, rules))